# Обучение V2 с нуля

Нужна CUDA GPU с BF16, например RTX 5090. T4 используется для инференса.
Ноутбук запускается из распакованного проекта. Подготовка полного датасета
и две эпохи обучения занимают значительное время; выполняйте по разделам.


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

root = next(p.resolve() for p in [Path.cwd(), Path.cwd().parent]
            if (p / 'v2/hw2_train_v2.py').is_file())
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=root, check=True)

run('-m', 'pip', 'install', '-r', root / 'requirements.txt')
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(torch.__version__, torch.cuda.get_device_name(0))


## Данные и проверки

Скачивается только исходный датасет на зафиксированной ревизии.
BPE из проекта используется повторно; train и validation кодируются
без усечения. Отбор 4096 validation-строк детерминирован и проверяется
относительно сохранённого исходного набора.


In [ ]:
run('download_all.py')
run('prepare_tokens.py')
run('-m', 'pytest', '-q', 'v2/tests')

validation = root / 'experiments/v2/validation4096.json'
if not validation.exists():
    run('v2/hw2_data_v2.py', '--cache', root / 'data/prepared/bpe8192',
        '--output', validation)
actual = json.loads(validation.read_text(encoding='utf-8'))
original = json.loads((root / 'evidence/validation4096.json').read_text(encoding='utf-8'))
assert actual['rows'] == original['rows']
print('Validation:', len(actual['rows']), actual['roles'])


## Две эпохи

Параметры взяты из конфигурации фактического полного запуска.
При наличии `last.pt` восстанавливаются optimizer, поток данных и RNG.
Исходные контрольные точки в проект не включены: здесь начинается новое обучение.


In [ ]:
config = root / 'v2/configs/v2_main_full_2ep.json'
directory = root / 'experiments/v2/v2_main_full_2ep'
args = ['v2/hw2_train_v2.py', '--root', root, '--config', config]
if (directory / 'last.pt').exists():
    args.append('--resume')
run(*args)


## Оценка и экспорт

Финальная сданная модель соответствует **последнему** шагу 111275.
Экспортируется `last.pt`. `best.pt` — отдельный выбор по dev и может
соответствовать другому шагу. AST agreement отличается от официального pass@1.


In [ ]:
checkpoint = directory / 'last.pt'
run('v2/hw2_eval_v2.py', '--checkpoint', checkpoint, '--precision', 'fp16',
    '--role', 'confirm', '--output', directory / 'confirm_fp16.json')
exports = list(directory.glob('t4_step*.zip'))
if not exports:
    run('v2/hw2_export_v2.py', '--checkpoint', checkpoint, '--root', root)
print('Экспорт:', list(directory.glob('t4_step*.zip')))
